# LLM evaluation: compare systems on a fixed toy set

Prerequisites: binary classification and confusion matrices. Estimated time: 30 minutes. Objectives: calculate task metrics, compare two systems on identical examples, and diagnose aggregate-score limitations.

A small binary moderation task illustrates why one aggregate metric is insufficient. The predictions are synthetic; the exercise concerns metric calculation and slice analysis, not a claim about model behavior.

## Dataset and baseline

Positive means the example should be flagged. The six cases include four positive and two negative labels. System A is a high-recall baseline with one false positive and one false negative.

In [ ]:
labels = [1, 1, 1, 1, 0, 0]
system_a = [1, 1, 1, 0, 1, 0]
assert len(labels) == len(system_a) == 6
print('examples:', len(labels), 'positive prevalence:', sum(labels) / len(labels))

## Confusion counts and metrics

Compute accuracy, precision, recall, and F1 from the same frozen labels and predictions. Precision is undefined when there are no predicted positives; this implementation returns zero and reports the counts so that convention is visible.

In [ ]:
def evaluate(y, pred):
    if len(y) != len(pred) or not y:
        raise ValueError('equal non-empty labels and predictions required')
    tp = sum(a == 1 and b == 1 for a,b in zip(y,pred))
    fp = sum(a == 0 and b == 1 for a,b in zip(y,pred))
    fn = sum(a == 1 and b == 0 for a,b in zip(y,pred))
    tn = sum(a == 0 and b == 0 for a,b in zip(y,pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'tp':tp,'fp':fp,'fn':fn,'tn':tn,'accuracy':(tp+tn)/len(y),'precision':precision,'recall':recall,'f1':f1}
metrics_a = evaluate(labels, system_a)
assert metrics_a == {'tp':3,'fp':1,'fn':1,'tn':1,'accuracy':4/6,'precision':0.75,'recall':0.75,'f1':0.75}
print('system A:', metrics_a)

## Candidate comparison and metric

System B is more conservative: it removes the false positive but misses two positive cases. Compare the systems on the same examples. Accuracy alone is tied, while recall and F1 expose a different operating point; threshold choice depends on the relative cost of missed flags and false alarms.

In [ ]:
system_b = [1, 1, 0, 0, 0, 0]
metrics_b = evaluate(labels, system_b)
assert metrics_b['accuracy'] == metrics_a['accuracy'] == 4/6
assert metrics_b['precision'] == 1.0 and metrics_b['recall'] == 0.5
assert round(metrics_b['f1'], 3) == 0.667
print('system B:', metrics_b, 'accuracy delta:', metrics_b['accuracy'] - metrics_a['accuracy'])

## Edge and failure cases

Aggregate scores can hide a slice regression, and a tiny test set has high uncertainty. Check subgroup and task categories, paired disagreements, label quality, contamination, and confidence intervals on a larger held-out set. Never tune a threshold on the final test set.

In [ ]:
# A no-positive-prediction case demonstrates the declared zero-precision convention.
all_negative = evaluate([1, 0], [0, 0])
assert all_negative['precision'] == 0.0 and all_negative['recall'] == 0.0
try:
    evaluate([1], [1, 0])
except ValueError:
    pass
else:
    raise AssertionError('misaligned examples must be rejected')

## Exercises

1. Assign a 5x cost to false negatives. Which system has lower observed cost, and what limitations remain?
2. Add a subgroup column and calculate recall for each group.
3. Design a human rubric and paired blinded comparison for open-ended answer quality; explain how you would estimate judge agreement.

## Summary

A useful evaluation freezes examples and labels, reports confusion counts and task metrics, compares candidate and baseline on identical cases, and inspects slices and uncertainty. LLM evaluation may also need human rubrics, safety checks, robustness, and cost/latency measures. See the [concept note](../concepts/32-evaluation.md) and the [practice assessment](../interview-prep/curriculum-practice.md#evaluation-for-llms).